# 2 · Laplacian graph drawing

Companion to **Section 4**. Drawing is the master problem with $\mathbf b=\mathbf 0$
in each coordinate, plus a constraint that prevents the collapse: anchors
(Tutte, a linear system) or a normalization (Hall and Koren, eigenvectors).

In [ ]:
import sys
sys.path.insert(0, "../code")          # the scripts that generate the paper's figures
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
np.set_printoptions(precision=4, suppress=True)


def draw(ax, P, edges, nodes_kw=None, title=None):
    """Draw a graph with node positions P (n x 2) and an edge list."""
    ax.add_collection(LineCollection([[P[i], P[j]] for i, j in edges],
                                     colors="0.75", linewidths=0.6, zorder=1))
    ax.scatter(P[:, 0], P[:, 1], s=14, zorder=2, **(nodes_kw or {}))
    ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)

In [ ]:
from scipy.linalg import eigh
from common import graph_drawing, spectra
from fig3_drawing import crossings
G, A, d, L, P_true, hull = graph_drawing()
lam, Q, *_ = spectra(L, d)
n, edges = len(d), list(G.edges())
print(f"n = {n}, m = {len(edges)}, degrees {d.min():.0f}..{d.max():.0f}, lambda_2 = {lam[1]:.4f}")

In [ ]:
P_hall = Q[:, 1:3]                                  # Hall: q_2, q_3 of L
mu, V = eigh(L, np.diag(d)); P_koren = V[:, 1:3]    # Koren: L v = mu D v
free = np.setdiff1d(np.arange(n), hull)             # Tutte: grounded system
P_tutte = P_true.copy()
P_tutte[free] = np.linalg.solve(L[np.ix_(free, free)], -L[np.ix_(free, hull)] @ P_true[hull])

fig, ax = plt.subplots(1, 4, figsize=(14, 3.6))
for a, (P, name) in zip(ax, [(P_true, "true positions"), (P_hall, "Hall"),
                             (P_koren, "degree-normalized"), (P_tutte, "Tutte")]):
    draw(a, P, edges, title=f"{name}: {crossings(edges, P)} crossings")
ax[3].scatter(*P_true[hull].T, marker="s", color="C3", zorder=3)
plt.show()

Hall's and Koren's layouts are computed from the graph alone and receive no
node positions. Tutte's layout is supplied with the coordinates of the ten
hull nodes; it is crossing-free because the graph is planar and 3-connected
and the anchors form a convex polygon.

## Tutte's layout is a leader–follower consensus

Iterate $\mathbf p_{\mathcal F}(k+1)=\mathbf p_{\mathcal F}(k)-\theta\,(L_{\mathcal{FF}}\mathbf p_{\mathcal F}(k)+L_{\mathcal{FA}}\mathbf p_{\mathcal A})$
from a random start: the error decays at the rate $1-\theta\lambda_{\min}(L_{\mathcal{FF}})$.

In [ ]:
LFF, LFA = L[np.ix_(free, free)], L[np.ix_(free, hull)]
lmin = np.linalg.eigvalsh(LFF).min()
theta = 1 / d.max()
p = np.random.default_rng(0).random((len(free), 2))
err = []
for k in range(200):
    err.append(np.linalg.norm(p - P_tutte[free]))
    p = p - theta * (LFF @ p + LFA @ P_true[hull])
plt.semilogy(err, label="anchored consensus")
plt.semilogy(err[0] * (1 - theta * lmin) ** np.arange(200), "--", label="(1 - theta lambda_min)^k")
plt.legend(); plt.xlabel("k"); plt.show()
print(f"lambda_min(L_FF) = {lmin:.4f}  vs  lambda_2(L) = {lam[1]:.4f}")

## How many leaders?

With a single anchor, Cauchy interlacing gives $\lambda_{\min}(L_{\mathcal{FF}})\le\lambda_2(L)$.
How many anchors does it take to make the anchored network *faster* than the
free one? Compare anchors taken along the hull with a greedy choice
(Exercise 5).

In [ ]:
def grounded_min(anchors):
    f = np.setdiff1d(np.arange(n), anchors)
    return np.linalg.eigvalsh(L[np.ix_(f, f)]).min()

print("hull anchors :", [round(float(grounded_min(hull[:k])), 3) for k in range(1, len(hull) + 1)])
greedy = []
for _ in range(5):
    greedy.append(max(set(range(n)) - set(greedy), key=lambda i: grounded_min(greedy + [i])))
    print(f"greedy, {len(greedy)} anchors: {grounded_min(greedy):.3f}")